#Chuẩn bị dataset


*   Không cần chạy vì có sẵn dataset ở drive rồi





In [1]:
import os
import sys
import io
import cv2
import json
import numpy as np
import shutil
from pycocotools.coco import COCO
from tqdm import tqdm
from google.colab import drive
# 1. CẤU HÌNH THÔNG SỐ VÀ ĐƯỜNG DẪN
TRAIN_LIMIT = 50000
VAL_LIMIT = 2500     # Lấy nửa đầu tập val2017
TEST_LIMIT = 2500    # Lấy nửa sau tập val2017
DRIVE_WORKSPACE = "/content/gdrive/MyDrive/PowerPaint_KhoaLuan"
REPO_DIR = os.path.join(DRIVE_WORKSPACE, "PowerPaint")
RAW_ZIP_PATH = "/content/gdrive/MyDrive/COCO_Raw_Data"
CLEAN_DATASET = "/content/coco_full_clean"
TEMP_EXTRACT = "/content/coco_temp"
COCO_URLS = {
    "train2017.zip": "http://images.cocodataset.org/zips/train2017.zip",
    "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
}
# 2. KHỞI TẠO MÔI TRƯỜNG
drive.mount('/content/gdrive')
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
os.makedirs(RAW_ZIP_PATH, exist_ok=True)
# Clone repo PowerPaint
if not os.path.exists(REPO_DIR):
    print("Clone PowerPaint")
    !git clone -b dev https://github.com/Hydro1112/PowerPaint "{REPO_DIR}"
sys.path.insert(0, REPO_DIR)
# 3. TẢI DATASET GỐC VÀ LƯU DRIVE
print("Download Dataset COCO 2017")
for file_name, url in COCO_URLS.items():
    dst_drive = os.path.join(RAW_ZIP_PATH, file_name)
    if not os.path.exists(dst_drive):
        print(f"Đang tải {file_name}")
        !wget -q -c {url} -O /content/{file_name}
        print(f"Đang lưu {file_name} vào Drive")
        !cp /content/{file_name} {dst_drive}
    else:
        if not os.path.exists(f"/content/{file_name}"):
            !cp {dst_drive} /content/
# 4. GIẢI NÉN DATASET
if not os.path.exists(TEMP_EXTRACT):
    print("\n Đang giải nén dataset COCO")
    !unzip -q /content/annotations_trainval2017.zip -d {TEMP_EXTRACT}
    !unzip -q /content/train2017.zip -d {TEMP_EXTRACT}
    !unzip -q /content/val2017.zip -d {TEMP_EXTRACT}
    !rm /content/train2017.zip /content/val2017.zip /content/annotations_trainval2017.zip

# 5. HÀM TRÍCH XUẤT CHUẨN HÓA (TRAIN/VAL/TEST)
def process_coco_split(split_name, annotation_mode, img_dir, limit, offset=0):
    print(f"\n Đang xử lý tập [{split_name.upper()}]")
    inst_file = f'{TEMP_EXTRACT}/annotations/instances_{annotation_mode}.json'
    capt_file = f'{TEMP_EXTRACT}/annotations/captions_{annotation_mode}.json'
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    coco_mask = COCO(inst_file)
    coco_text = COCO(capt_file)
    sys.stdout = old_stdout
    img_out = os.path.join(CLEAN_DATASET, split_name, "images")
    mask_out = os.path.join(CLEAN_DATASET, split_name, "masks")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(mask_out, exist_ok=True)
    img_ids = coco_mask.getImgIds()
    img_ids.sort()
    img_ids_target = img_ids[offset : offset + limit]
    metadata = []
    for img_id in tqdm(img_ids_target, desc=f"Lọc & Tạo Mask {split_name}"):
        img_info = coco_mask.loadImgs(img_id)[0]
        file_name = img_info['file_name']
        src_img_path = os.path.join(img_dir, file_name)
        if not os.path.exists(src_img_path):
            continue
        ann_ids = coco_mask.getAnnIds(imgIds=img_id)
        anns = coco_mask.loadAnns(ann_ids)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, coco_mask.annToMask(ann) * 255)
        if np.max(mask) == 0:
            continue
        ann_text_ids = coco_text.getAnnIds(imgIds=img_id)
        text_anns = coco_text.loadAnns(ann_text_ids)
        caption = text_anns[0]['caption'] if len(text_anns) > 0 else ""
        if caption:
            dst_img_path = os.path.join(img_out, file_name)
            dst_mask_path = os.path.join(mask_out, file_name.replace('.jpg', '.png'))
            shutil.copy(src_img_path, dst_img_path)
            cv2.imwrite(dst_mask_path, mask)
            metadata.append({
                "image_path": f"{split_name}/images/{file_name}",
                "mask_path": f"{split_name}/masks/{file_name.replace('.jpg', '.png')}",
                "caption": caption.strip()
            })
    with open(os.path.join(CLEAN_DATASET, split_name, "metadata.json"), "w") as f:
        json.dump(metadata, f, indent=4)
    print(f"Tập {split_name.upper()} có {len(metadata)} mẫu hợp lệ")
# THỰC THI CHO 3 TẬP
process_coco_split("train", "train2017", f'{TEMP_EXTRACT}/train2017', TRAIN_LIMIT, offset=0)
process_coco_split("val", "val2017", f'{TEMP_EXTRACT}/val2017', VAL_LIMIT, offset=0)
process_coco_split("test", "val2017", f'{TEMP_EXTRACT}/val2017', TEST_LIMIT, offset=VAL_LIMIT)
# 6. NÉN VÀ ĐẨY LÊN DRIVE
!zip -r -q /content/coco_full_clean.zip {CLEAN_DATASET}
print("Đưa dataset lên Drive")
!cp /content/coco_full_clean.zip {RAW_ZIP_PATH}/
print("Done")

Mounted at /content/gdrive
Download Dataset COCO 2017
Đang tải train2017.zip
Đang lưu train2017.zip vào Drive
Đang tải val2017.zip
Đang lưu val2017.zip vào Drive
Đang tải annotations_trainval2017.zip
Đang lưu annotations_trainval2017.zip vào Drive

 Đang giải nén dataset COCO

 Đang xử lý tập [TRAIN]


Lọc & Tạo Mask train: 100%|██████████| 50000/50000 [08:36<00:00, 96.78it/s]


Tập TRAIN có 49551 mẫu hợp lệ

 Đang xử lý tập [VAL]


Lọc & Tạo Mask val: 100%|██████████| 2500/2500 [00:24<00:00, 100.11it/s]


Tập VAL có 2477 mẫu hợp lệ

 Đang xử lý tập [TEST]


Lọc & Tạo Mask test: 100%|██████████| 2500/2500 [00:18<00:00, 137.43it/s]


Tập TEST có 2475 mẫu hợp lệ
Đưa dataset lên Drive
Done


#Cài đặt thư viện

In [1]:
# Python 3.10
import os
from google.colab import drive
drive.mount('/content/gdrive')
REPO_DIR = "/content/gdrive/MyDrive/PowerPaint_KhoaLuan/PowerPaint"
!apt-get update -y > /dev/null
!apt-get install python3.10 python3.10-distutils -y > /dev/null
!wget -q https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py > /dev/null
# Install requirements
os.chdir(REPO_DIR)
!python3.10 -m pip install -r requirements/requirements.txt \
    --extra-index-url https://download.pytorch.org/whl/cu118
print("Done")

Mounted at /content/gdrive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 46.8 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 166.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of xformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.3/905.3 MB 37.6 MB/s  0:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 142.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#Train

In [2]:
import os
import sys
import glob
import json
import torch
import gc
from google.colab import drive
#THÔNG SỐ HYPERPARAMETERS
COCO_LIMIT = 5000               # Số lượng ảnh muốn lấy từ tập TRAIN
NUM_EPOCHS = 50                 # Số vòng lặp qua dataset
BATCH_SIZE = 2                  # Số ảnh/batch (Tăng nếu GPU VRAM rảnh)
GRADIENT_ACC = 8                # Tích lũy Gradient (Batch thực = BATCH_SIZE * GRADIENT_ACC)
LEARNING_RATE = "5e-6"          # Tốc độ học (Dạng string)
CHECKPOINT_STEPS = 500          # Lưu weights sau mỗi 500 bước
MIXED_PRECISION = "fp16"        # Tiết kiệm VRAM
#KHỞI TẠO ĐƯỜNG DẪN & MÔI TRƯỜNG
drive.mount('/content/gdrive')
DRIVE_WORKSPACE = "/content/gdrive/MyDrive/PowerPaint_KhoaLuan"
REPO_DIR = os.path.join(DRIVE_WORKSPACE, "PowerPaint")
OUTPUT_DIR = os.path.join(DRIVE_WORKSPACE, "output_models")
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
#Khai báo đường dẫn Repo vào hệ thống
sys.path.insert(0, REPO_DIR)
# Lọc log cảnh báo rác
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"
#CHUẨN BỊ DỮ LIỆU
print("\n Prepare Data Train")
CLEAN_ZIP_PATH = "/content/gdrive/MyDrive/COCO_Raw_Data/coco_full_clean.zip"
BASE_DATA_DIR = "/content/coco_full_clean"
DATASET_DIR = os.path.join(BASE_DATA_DIR, "train")
if not os.path.exists(os.path.join(DATASET_DIR, "metadata.json")):
    !cp "{CLEAN_ZIP_PATH}" /content/temp_data.zip
    !unzip -q /content/temp_data.zip -d /content/
    !rm /content/temp_data.zip
    nested_dir = "/content/content/coco_full_clean"
    if os.path.exists(nested_dir):
        os.makedirs(BASE_DATA_DIR, exist_ok=True)
        !mv {nested_dir}/* {BASE_DATA_DIR}/
        !rm -rf /content/content
with open(os.path.join(DATASET_DIR, "metadata.json"), "r") as f:
    full_metadata = json.load(f)
actual_metadata = full_metadata[:COCO_LIMIT]
for item in actual_metadata:
    img_filename = os.path.basename(item["image_path"])
    mask_filename = os.path.basename(item["mask_path"])
    item["image_path"] = os.path.join(DATASET_DIR, "images", img_filename)
    item["mask_path"] = os.path.join(DATASET_DIR, "masks", mask_filename)
with open(os.path.join(DATASET_DIR, "metadata_run.json"), "w") as f:
    json.dump(actual_metadata, f, indent=4)
actual_count = len(actual_metadata)
print(f"Chọn {actual_count} mẫu Data hợp lệ cho quá trình train.")
#KHỞI CHẠY TRAINING
#Dọn dẹp VRAM trước khi chạy
torch.cuda.empty_cache(); gc.collect()
#Logic tìm và chạy tiếp từ file lưu (Resume)
ckpt_list = glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*"))
resume_arg = ""

print("\nKiểm tra Checkpoint...")
if ckpt_list:
    latest_ckpt = max(ckpt_list, key=os.path.getctime)
    resume_arg = f'--resume_from_checkpoint="{os.path.basename(latest_ckpt)}"'
    print(f"tìm thấy bản lưu{latest_ckpt}")
else:
    print("Không tìm thấy bản lưu")
os.chdir(REPO_DIR)
!python3.10 -m accelerate.commands.launch train_ppt2_bn.py \
    --config "configs/config_coco.yaml" \
    {resume_arg}

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

 Prepare Data Train
Chọn 5000 mẫu Data hợp lệ cho quá trình train.

Kiểm tra Checkpoint...
Không tìm thấy bản lưu
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
05/09/2026 07:36:54 - INFO - __main__ - [RANK 0] Distributed environment: NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

config.json: 100% 743/743 [00:00<00:00, 4.26MB/s]
diffusion_pytorch_model.safetensors: 100% 3.44G/3.44G [00:09<00:00, 364MB/s] 
{'transformer_layers_per_block', 'use_linear_projection', 'time_emb